In [1]:
# ============================================================
# Cell 0 — Install dependencies
# Internet OFF: all packages must come from Kaggle base image.
# CPU session is sufficient for this stage.
# ============================================================
!pip install -q underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 60.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.5 MB/s eta 0:00:00


In [2]:
# ============================================================
# Cell 1 — Paths
# INPUT_STAGE1: the Kaggle Dataset saved from Stage 1
#   -> Name it exactly: legal-llm-stage1-hf-datasets
#   -> Add via Kaggle UI 'Add Data' before running this notebook
# ============================================================
import os, sys

OUTPUT_BASE   = "/kaggle/working"
PROCESSED_DIR = f"{OUTPUT_BASE}/data_processed"
LOG_DIR       = f"{OUTPUT_BASE}/logs"

# Input: Stage 1 HF datasets (from saved Kaggle dataset)
# Adjust this path to match the actual Kaggle dataset slug
STAGE1_HF     = "/kaggle/input/datasets/giahuyvotran/legal-llm-stage1-hf-datasets/data_raw/huggingface"

# Input: Original Kaggle datasets (same as Stage 1)
ZALO_INPUT    = "/kaggle/input/datasets/giahuyvotran/legal-llm-stage1-hf-datasets/data_raw/zalo"
QUANGBUT_INPUT= "/kaggle/input/datasets/giahuyvotran/legal-llm-stage1-hf-datasets/data_raw/quangbut"
TUONG_INPUT   = "/kaggle/input/datasets/giahuyvotran/legal-llm-stage1-hf-datasets/data_raw/tuongdang"

# Output directories
CPT_DIR       = f"{PROCESSED_DIR}/cpt"
SFT_DIR       = f"{PROCESSED_DIR}/sft"
DPO_DIR       = f"{PROCESSED_DIR}/dpo"
RAG_DIR       = f"{PROCESSED_DIR}/rag"
EVAL_DIR      = f"{PROCESSED_DIR}/eval"

for d in [PROCESSED_DIR, LOG_DIR, CPT_DIR, SFT_DIR, DPO_DIR, RAG_DIR, EVAL_DIR]:
    os.makedirs(d, exist_ok=True)

print("Python:", sys.version)
print("Stage1 HF path exists:", os.path.exists(STAGE1_HF))
print("Zalo input exists:    ", os.path.exists(ZALO_INPUT))
print("Quangbut input exists:", os.path.exists(QUANGBUT_INPUT))
print("Directories created.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Stage1 HF path exists: True
Zalo input exists:     True
Quangbut input exists: True
Directories created.


In [3]:
# ============================================================
# Cell 2 — Utility functions: text cleaning
# Applied to ALL datasets before saving.
# ============================================================
import re
import unicodedata
import json

# Mandatory disclaimer appended to every SFT/DPO output
DISCLAIMER = (
    "\n\nLưu ý: Nội dung trên chỉ mang tính chất tham khảo, "
    "không phải tư vấn pháp lý chính thức. "
    "Hãy tham khảo luật sư có chuyên môn để được tư vấn cụ thể."
)

INSTRUCTION_SYSTEM = (
    "Bạn là trợ lý pháp lý chuyên về luật Việt Nam. "
    "Hãy trả lời câu hỏi dựa trên các quy định pháp luật hiện hành, "
    "trích dẫn rõ điều khoản và văn bản pháp lý liên quan."
)


def normalize_unicode(text: str) -> str:
    """NFC normalization — essential for Vietnamese diacritics consistency."""
    return unicodedata.normalize("NFC", text)


def deidentify_pii(text: str) -> str:
    """Remove Vietnamese personal identifiers."""
    # Vietnamese full names (Họ + Tên đệm + Tên)
    text = re.sub(
        r'\b(Nguyễn|Trần|Lê|Phạm|Hoàng|Huỳnh|Phan|Vũ|Võ|Đặng|Bùi|Đỗ|Hồ|Ngô|Dương)'
        r'\s+[A-ZÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚĂĐĨŨƠƯẠẶ][a-zàáâãèéêìíòóôõùúăđĩũơưạặ]+'
        r'(?:\s+[A-ZÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚĂĐĨŨƠƯẠẶ][a-zàáâãèéêìíòóôõùúăđĩũơưạặ]+)?',
        '[HO_TEN]', text
    )
    text = re.sub(r'\b\d{9,12}\b', '[SO_GIAY_TO]', text)  # CCCD/CMND
    text = re.sub(r'\b(0|\+84)[3-9]\d{8}\b', '[SO_DIEN_THOAI]', text)
    return text


def normalize_legal_citation(text: str) -> str:
    """Capitalize standard legal citation patterns."""
    # Ensure 'điều' -> 'Điều', 'khoản' -> 'Khoản', 'điểm' -> 'Điểm'
    text = re.sub(r'\bđiều\s+(\d+)', r'Điều \1', text)
    text = re.sub(r'\bkhoản\s+(\d+)', r'Khoản \1', text)
    text = re.sub(r'\bđiểm\s+([a-zđ])', r'Điểm \1', text)
    return text


def clean_text(text: str) -> str:
    """Full cleaning pipeline for a single text string."""
    text = normalize_unicode(text)
    text = normalize_legal_citation(text)
    text = re.sub(r'\s+', ' ', text).strip()   # collapse whitespace
    return text


def passes_quality_filter(text: str, min_chars: int = 80, max_chars: int = 8192) -> bool:
    """Reject texts that are too short, too long, or mostly non-alphanumeric."""
    if not min_chars <= len(text) <= max_chars:
        return False
    alpha_ratio = sum(1 for c in text if c.isalpha()) / max(len(text), 1)
    return alpha_ratio >= 0.4


def write_jsonl(records: list, path: str) -> int:
    """Write a list of dicts to JSONL. Returns count written."""
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return len(records)


print("Utility functions loaded.")
# Quick sanity checks
assert normalize_unicode("Điều") == "Điều"
assert passes_quality_filter("x" * 50) == False
assert passes_quality_filter("Điều này quy định về trách nhiệm pháp lý của các bên.", min_chars=30) == True
long_test = "Điều này quy định về trách nhiệm pháp lý của các bên khi tham gia giao dịch dân sự, thương mại và hợp đồng theo pháp luật Việt Nam."
assert passes_quality_filter(long_test) == True
print("Sanity checks passed.")

Utility functions loaded.
Sanity checks passed.


In [4]:
# ============================================================
# Cell 3 — Zalo legal_corpus.json
# Output A: cpt/cpt_zalo.jsonl       (plaintext for CPT)
# Output B: rag/rag_zalo_passages.jsonl (passages with metadata)
# Structure: {law_id, articles: [{article_id, title, text}]}
# ============================================================
import json, os

ZALO_CORPUS = f"{ZALO_INPUT}/legal_corpus.json"

print(f"Loading {ZALO_CORPUS} ...")
with open(ZALO_CORPUS, "r", encoding="utf-8") as f:
    corpus_data = json.load(f)

cpt_zalo    = []
rag_zalo    = []
skipped     = 0

for doc in corpus_data:
    law_id = doc.get("law_id", "unknown")

    for article in doc.get("articles", []):
        raw_text = article.get("text", "").strip()
        title    = article.get("title", "").strip()
        art_id   = article.get("article_id", "")

        cleaned = clean_text(raw_text)

        if not passes_quality_filter(cleaned, min_chars=50):
            skipped += 1
            continue

        # CPT: combine title + text as plain document
        cpt_text = f"{title}\n{cleaned}" if title else cleaned
        cpt_zalo.append({"text": cpt_text, "source": "zalo", "law_id": law_id})

        # RAG passage: with full metadata for retrieval filtering
        rag_zalo.append({
            "id":   f"{law_id}_dieu{art_id}",
            "text": cpt_text,
            "metadata": {
                "van_ban":      law_id,
                "dieu":         title,
                "article_id":   art_id,
                "linh_vuc":     "unknown",          # enriched in RAG stage
                "nam_ban_hanh": "unknown",
                "source":       "zalo_legal_corpus"
            }
        })

# Save
n_cpt = write_jsonl(cpt_zalo, f"{CPT_DIR}/cpt_zalo.jsonl")
n_rag = write_jsonl(rag_zalo, f"{RAG_DIR}/rag_zalo_passages.jsonl")

print(f"  Documents processed : {len(corpus_data):,}")
print(f"  Articles accepted   : {n_cpt:,}")
print(f"  Articles skipped    : {skipped:,}")
print(f"  CPT -> {CPT_DIR}/cpt_zalo.jsonl ({os.path.getsize(CPT_DIR+'/cpt_zalo.jsonl')/1024**2:.2f} MB)")
print(f"  RAG -> {RAG_DIR}/rag_zalo_passages.jsonl ({os.path.getsize(RAG_DIR+'/rag_zalo_passages.jsonl')/1024**2:.2f} MB)")

# Sample
print("\nSample CPT record:")
print(json.dumps(cpt_zalo[0], ensure_ascii=False)[:300])
print("\nSample RAG record (metadata):")
print(json.dumps(rag_zalo[0]["metadata"], ensure_ascii=False))

Loading /kaggle/input/datasets/giahuyvotran/legal-llm-stage1-hf-datasets/data_raw/zalo/legal_corpus.json ...
  Documents processed : 3,271
  Articles accepted   : 60,522
  Articles skipped    : 903
  CPT -> /kaggle/working/data_processed/cpt/cpt_zalo.jsonl (99.70 MB)
  RAG -> /kaggle/working/data_processed/rag/rag_zalo_passages.jsonl (112.41 MB)

Sample CPT record:
{"text": "Điều 1. Phạm vi áp dụng\nThông tư này hướng dẫn tuần tra, canh gác bảo vệ đê Điều trong mùa lũ đối với các tuyến đê sông được phân loại, phân cấp theo quy định tại Điều 4 của Luật Đê Điều.", "source": "zalo", "law_id": "01/2009/tt-bnn"}

Sample RAG record (metadata):
{"van_ban": "01/2009/tt-bnn", "dieu": "Điều 1. Phạm vi áp dụng", "article_id": "1", "linh_vuc": "unknown", "nam_ban_hanh": "unknown", "source": "zalo_legal_corpus"}


In [5]:
# ============================================================
# Cell 4 — Vietnamese Legal QA (thangvip)
# Schema: doc_name, doc_type_name, article_content,
#         generated_qa_pairs (list of {question, answer}),
#         generation_time
# Output A: sft/sft_vn_legal_qa.jsonl   (instruction format)
# Output B: cpt/cpt_vn_legal_qa.jsonl   (raw article_content for CPT)
# ============================================================
import json, os

VN_QA_TRAIN = f"{STAGE1_HF}/vietnamese-legal-qa/train.jsonl"

sft_qa  = []
cpt_qa  = []
skipped = 0

with open(VN_QA_TRAIN, "r", encoding="utf-8") as f:
    for line in f:
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue

        doc_name      = record.get("doc_name", "")
        article_text  = clean_text(record.get("article_content", ""))
        qa_pairs_raw  = record.get("generated_qa_pairs", [])

        # CPT: raw article content
        if passes_quality_filter(article_text, min_chars=100):
            cpt_qa.append({"text": article_text, "source": "vn_legal_qa", "doc_name": doc_name})

        # Parse generated_qa_pairs — may be a JSON string or a list
        if isinstance(qa_pairs_raw, str):
            try:
                qa_pairs = json.loads(qa_pairs_raw)
            except json.JSONDecodeError:
                skipped += 1
                continue
        elif isinstance(qa_pairs_raw, list):
            qa_pairs = qa_pairs_raw
        else:
            skipped += 1
            continue

        for qa in qa_pairs:
            if not isinstance(qa, dict):
                continue
            # Accept both 'question'/'answer' and 'Question'/'Answer'
            question = qa.get("question") or qa.get("Question", "")
            answer   = qa.get("answer")   or qa.get("Answer",   "")

            if not question or not answer:
                continue

            question = clean_text(str(question))
            answer   = clean_text(str(answer))

            if not passes_quality_filter(answer, min_chars=30):
                skipped += 1
                continue

            sft_qa.append({
                "instruction": INSTRUCTION_SYSTEM,
                "input":       f"Câu hỏi: {question}\n\nNguồn tham chiếu:\n{article_text[:1500]}",
                "output":      answer + DISCLAIMER,
                "source":      "vn_legal_qa",
                "doc_name":    doc_name
            })

n_sft = write_jsonl(sft_qa, f"{SFT_DIR}/sft_vn_legal_qa.jsonl")
n_cpt = write_jsonl(cpt_qa, f"{CPT_DIR}/cpt_vn_legal_qa.jsonl")

print(f"  Source records    : 9,715")
print(f"  SFT samples saved : {n_sft:,}  -> sft_vn_legal_qa.jsonl")
print(f"  CPT articles saved: {n_cpt:,}  -> cpt_vn_legal_qa.jsonl")
print(f"  Skipped           : {skipped:,}")

if sft_qa:
    print("\nSample SFT record:")
    s = sft_qa[0]
    print(f"  instruction: {s['instruction'][:60]}...")
    print(f"  input[:100]: {s['input'][:100]}...")
    print(f"  output[:100]: {s['output'][:100]}...")

  Source records    : 9,715
  SFT samples saved : 29,141  -> sft_vn_legal_qa.jsonl
  CPT articles saved: 9,377  -> cpt_vn_legal_qa.jsonl
  Skipped           : 4

Sample SFT record:
  instruction: Bạn là trợ lý pháp lý chuyên về luật Việt Nam. Hãy trả lời c...
  input[:100]: Câu hỏi: Theo Điều 5 Luật Địa chất và Khoáng sản, hội nhập và hợp tác quốc tế về địa chất, khoáng sả...
  output[:100]: Theo Khoản 1 Điều 5 Luật Địa chất và Khoáng sản, hội nhập và hợp tác quốc tế về địa chất, khoáng sản...


In [6]:
# ============================================================
# Cell 5 — Vietnamese Legal Chat (luanngo)
# Schema: conversations (list of {from: human/gpt, value: str})
# Output: sft/sft_vn_legal_chat.jsonl (instruction format)
# ============================================================
import json, os

VN_CHAT = f"{STAGE1_HF}/vietnamese-legal-chat/train.jsonl"

sft_chat = []
skipped  = 0

with open(VN_CHAT, "r", encoding="utf-8") as f:
    for line in f:
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue

        conversations = record.get("conversations", [])
        if not isinstance(conversations, list) or len(conversations) < 2:
            skipped += 1
            continue

        # Convert ShareGPT format: alternating human / gpt turns
        # Pair each human turn with the immediately following gpt turn
        i = 0
        while i < len(conversations) - 1:
            turn_human = conversations[i]
            turn_gpt   = conversations[i + 1]

            role_h = turn_human.get("from", "").lower()
            role_g = turn_gpt.get("from", "").lower()

            if role_h not in ("human", "user") or role_g not in ("gpt", "assistant"):
                i += 1
                continue

            question = clean_text(str(turn_human.get("value", "")))
            answer   = clean_text(str(turn_gpt.get("value", "")))

            if not passes_quality_filter(answer, min_chars=30):
                i += 2
                skipped += 1
                continue

            sft_chat.append({
                "instruction": INSTRUCTION_SYSTEM,
                "input":       question,
                "output":      answer + DISCLAIMER,
                "source":      "vn_legal_chat"
            })
            i += 2

n_sft = write_jsonl(sft_chat, f"{SFT_DIR}/sft_vn_legal_chat.jsonl")

print(f"  Conversations     : 3,537")
print(f"  SFT pairs saved   : {n_sft:,}  -> sft_vn_legal_chat.jsonl")
print(f"  Skipped           : {skipped:,}")

if sft_chat:
    print("\nSample SFT record:")
    print(f"  input[:120] : {sft_chat[0]['input'][:120]}")
    print(f"  output[:120]: {sft_chat[0]['output'][:120]}")

  Conversations     : 3,537
  SFT pairs saved   : 2,651  -> sft_vn_legal_chat.jsonl
  Skipped           : 886

Sample SFT record:
  input[:120] : Dưới đây là các câu hỏi trắc nghiệm (có đáp án) về legal multiple choice. Vui lòng chọn đáp án phù hợp nhất cho câu hỏi 
  output[:120]: Để đảm bảo quyền tự do, bí mật trong bầu cử của mỗi người dân.

Lưu ý: Nội dung trên chỉ mang tính chất tham khảo, không


In [7]:
# ============================================================
# Cell 6 — Large VN Legal Queries (phamson02)
# Schema: passage_id, domain, title, header, context, aspect, query
# Output A: rag/rag_large_queries_passages.jsonl (for vector index)
# Output B: rag/rag_large_queries_pairs.jsonl    (query-passage pairs for retrieval training)
# Output C: cpt/cpt_large_queries.jsonl          (context field for CPT)
# ============================================================
import json, os

LARGE_QUERIES = f"{STAGE1_HF}/large-vi-legal-queries/train.jsonl"

rag_passages  = []
rag_pairs     = []
cpt_queries   = []
skipped       = 0
seen_passages = set()   # deduplicate by passage_id

with open(LARGE_QUERIES, "r", encoding="utf-8") as f:
    for line in f:
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        pid     = rec.get("passage_id", "")
        domain  = rec.get("domain", "")
        title   = rec.get("title", "")
        header  = rec.get("header", "")
        context = clean_text(rec.get("context", ""))
        query   = clean_text(rec.get("query", ""))

        if not passes_quality_filter(context, min_chars=50):
            skipped += 1
            continue

        # Deduplicated RAG passages (many queries share the same passage)
        if pid not in seen_passages:
            seen_passages.add(pid)
            passage_text = f"{title}\n{header}\n{context}".strip() if (title or header) else context
            rag_passages.append({
                "id":   pid,
                "text": passage_text,
                "metadata": {
                    "domain":  domain,
                    "title":   title,
                    "source":  "large_vi_legal_queries"
                }
            })
            # CPT: passage text
            cpt_queries.append({"text": passage_text, "source": "large_vi_legal_queries"})

        # Query-passage pair for retrieval training
        if query:
            rag_pairs.append({
                "query":      query,
                "passage_id": pid,
                "domain":     domain
            })

n_pass = write_jsonl(rag_passages, f"{RAG_DIR}/rag_large_queries_passages.jsonl")
n_pair = write_jsonl(rag_pairs,    f"{RAG_DIR}/rag_large_queries_pairs.jsonl")
n_cpt  = write_jsonl(cpt_queries,  f"{CPT_DIR}/cpt_large_queries.jsonl")

print(f"  Source rows       : 500,000")
print(f"  Unique passages   : {n_pass:,}  -> rag_large_queries_passages.jsonl")
print(f"  Query-passage pairs: {n_pair:,} -> rag_large_queries_pairs.jsonl")
print(f"  CPT contexts      : {n_cpt:,}   -> cpt_large_queries.jsonl")
print(f"  Skipped           : {skipped:,}")

  Source rows       : 500,000
  Unique passages   : 138,268  -> rag_large_queries_passages.jsonl
  Query-passage pairs: 499,901 -> rag_large_queries_pairs.jsonl
  CPT contexts      : 138,268   -> cpt_large_queries.jsonl
  Skipped           : 99


In [8]:
# ============================================================
# Cell 7 — ViBidLQA_v1 (min_chars lowered to 20 for DPO)
# Schema: context, question, answer
# Output A: eval/eval_gold.jsonl
# Output B: sft/sft_vibidlqa.jsonl
# Output C: dpo/dpo_vibidlqa.jsonl (min_chars=20)
# ============================================================
import json, os, re

CITATION_PATTERN = re.compile(r'(Điều\s+\d+|Khoản\s+\d+|Điểm\s+[a-zđ])', re.IGNORECASE)

def extract_citation_hint(context: str) -> str:
    matches = CITATION_PATTERN.findall(context)
    return matches[0] if matches else ""

def build_sft_record(q, a, ctx, src):
    citation = extract_citation_hint(ctx)
    cite_str = f" (Căn cứ: {citation})" if citation else ""
    return {
        "instruction": INSTRUCTION_SYSTEM,
        "input":       f"Câu hỏi: {q}\n\nNguồn tham chiếu:\n{ctx[:1500]}",
        "output":      a + cite_str + DISCLAIMER,
        "source":      src
    }

def build_dpo_pair(q, a, ctx):
    citation = extract_citation_hint(ctx)
    cite_str = f" Căn cứ: {citation}." if citation else ""
    chosen   = a + cite_str + DISCLAIMER
    sentences = a.split('. ')
    rejected  = '. '.join(sentences[:-1]) + '.' if len(sentences) > 1 else a[:max(1, len(a)//2)]
    return {
        "prompt":   f"Câu hỏi: {q}\n\nNguồn:\n{ctx[:800]}",
        "chosen":   chosen,
        "rejected": rejected
    }

eval_gold = []
sft_vibid = []
dpo_vibid = []

for split_name, output_role in [("train", "sft+dpo"), ("validation", "eval"), ("test", "eval")]:
    path = f"{STAGE1_HF}/ViBidLQA_v1/{split_name}.jsonl"
    if not os.path.exists(path):
        print(f"  [SKIP] {path}")
        continue

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            ctx = clean_text(rec.get("context", ""))
            q   = clean_text(rec.get("question", ""))
            a   = clean_text(rec.get("answer", ""))

            if not q or not a:
                continue

            eval_gold.append({
                "context": ctx, "question": q, "answer": a,
                "split": split_name, "source": "ViBidLQA_v1"
            })

            if split_name == "train":
                sft_vibid.append(build_sft_record(q, a, ctx, "ViBidLQA_v1"))
                if passes_quality_filter(a, min_chars=20):
                    dpo_vibid.append(build_dpo_pair(q, a, ctx))

n_eval = write_jsonl(eval_gold, f"{EVAL_DIR}/eval_gold.jsonl")
n_sft  = write_jsonl(sft_vibid, f"{SFT_DIR}/sft_vibidlqa.jsonl")
n_dpo  = write_jsonl(dpo_vibid, f"{DPO_DIR}/dpo_vibidlqa.jsonl")

print(f"  Gold eval set  : {n_eval:,}  -> eval/eval_gold.jsonl")
print(f"  SFT samples    : {n_sft:,}   -> sft/sft_vibidlqa.jsonl")
print(f"  DPO pairs      : {n_dpo:,}   -> dpo/dpo_vibidlqa.jsonl")

if dpo_vibid:
    print("\nSample DPO pair:")
    p = dpo_vibid[0]
    print(f"  prompt[:80]  : {p['prompt'][:80]}")
    print(f"  chosen[:80]  : {p['chosen'][:80]}")
    print(f"  rejected[:80]: {p['rejected'][:80]}")


  Gold eval set  : 3,013  -> eval/eval_gold.jsonl
  SFT samples    : 1,928   -> sft/sft_vibidlqa.jsonl
  DPO pairs      : 1,424   -> dpo/dpo_vibidlqa.jsonl

Sample DPO pair:
  prompt[:80]  : Câu hỏi: Những cách tham vấn thị trường là gì, theo Khoản 1 Điều 56 của Luật Đấu
  chosen[:80]  : các phương tiện thông tin phù hợp; nghiên cứu catalô và các tài liệu giới thiệu 
  rejected[:80]: các phương tiện thông tin phù hợp; nghiên cứu catalô và các tài liệu giới thiệu 


In [9]:
# ============================================================
# Cell 7b — Supplement DPO from VN Legal QA
# ViBidLQA only produced 781 DPO pairs (< 2,000 target).
#
# Strategy:
#   - Read sft_vn_legal_qa.jsonl (saved by Cell 4)
#   - chosen   = full answer (original SFT output)
#   - rejected = first sentence only (stripped legal reasoning)
#   - Merge ViBidLQA + VN Legal QA into dpo_train.jsonl
# ============================================================

import json
import os
import random

VN_QA_SFT = f"{SFT_DIR}/sft_vn_legal_qa.jsonl"
MAX_SUPP  = 5_000  # cap to avoid source imbalance

dpo_supp = []
supp_skipped = 0

if not os.path.exists(VN_QA_SFT):
    print(f"[ERROR] Missing file: {VN_QA_SFT}")
else:
    with open(VN_QA_SFT, "r", encoding="utf-8") as f:
        for line in f:
            if len(dpo_supp) >= MAX_SUPP:
                break

            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            full_answer = rec.get("output", "").strip()
            if not full_answer:
                supp_skipped += 1
                continue

            # -------------------------
            # Strip disclaimer section
            # -------------------------
            core = full_answer.split("\n\nLưu ý:")[0].strip()

            sentences = [
                s.strip()
                for s in core.split(". ")
                if s.strip()
            ]

            if len(sentences) < 2 or not passes_quality_filter(core, min_chars=50):
                supp_skipped += 1
                continue

            # rejected: only first sentence (weaker reasoning)
            rejected = sentences[0]
            if not rejected.endswith("."):
                rejected += "."

            dpo_supp.append({
                "prompt":   rec.get("input", "")[:800],
                "chosen":   full_answer,
                "rejected": rejected
            })

# -------------------------
# Write supplement file
# -------------------------

n_supp = write_jsonl(
    dpo_supp,
    f"{DPO_DIR}/dpo_vn_legal_qa.jsonl"
)

print(f"VN Legal QA DPO supplement: {n_supp:,} pairs -> dpo_vn_legal_qa.jsonl")
print(f"Skipped: {supp_skipped:,}")

# ============================================================
# Merge ViBidLQA + VN Legal QA into dpo_train.jsonl
# ============================================================

all_dpo = []

for src_path in [
    f"{DPO_DIR}/dpo_vibidlqa.jsonl",
    f"{DPO_DIR}/dpo_vn_legal_qa.jsonl",
]:
    if not os.path.exists(src_path):
        print(f"[SKIP] {src_path}")
        continue

    with open(src_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                all_dpo.append(json.loads(line))
            except:
                continue

random.seed(42)
random.shuffle(all_dpo)

n_merged = write_jsonl(
    all_dpo,
    f"{DPO_DIR}/dpo_train.jsonl"
)

print(f"\nMerged DPO total: {n_merged:,} pairs -> dpo_train.jsonl")
print("PASS" if n_merged >= 2000 else "STILL BELOW 2000 — check sources")

VN Legal QA DPO supplement: 5,000 pairs -> dpo_vn_legal_qa.jsonl
Skipped: 1,687

Merged DPO total: 6,424 pairs -> dpo_train.jsonl
PASS


In [10]:
# ============================================================
# Cell 8 — Quangbut Vietnamese Legal (UTF-16 encoded CSVs)
# Files: vbpl.csv, legal_truncated_corpus.csv, sent_truncated_*.csv
# FIX: encoding='utf-16' (0xff 0xfe BOM detected in Stage 1)
# Output: cpt/cpt_quangbut.jsonl (supplementary CPT text)
# Strategy: sample 50k rows per large CSV to avoid OOM
# ============================================================
import pandas as pd
import json, os, glob

MAX_ROWS_PER_FILE = 50_000   # Adjust up if memory allows
cpt_quangbut = []
skipped = 0


def get_text_columns(df: pd.DataFrame) -> list:
    """Return columns with average string length > 50 chars (likely text, not IDs)."""
    text_cols = []
    for col in df.select_dtypes(include="object").columns:
        sample_len = df[col].dropna().astype(str).str.len().mean()
        if sample_len > 50:
            text_cols.append(col)
    return text_cols


csv_files = [
    f"{QUANGBUT_INPUT}/vbpl.csv",
    f"{QUANGBUT_INPUT}/legal_truncated_corpus.csv",
    f"{QUANGBUT_INPUT}/sent_truncated_vbpl_update.csv",
    f"{QUANGBUT_INPUT}/sent_truncated_vbpl_legal_only.csv",
    f"{TUONG_INPUT}/vbpl_crawl_2.csv",
]

for csv_path in csv_files:
    if not os.path.exists(csv_path):
        print(f"  [SKIP] {os.path.basename(csv_path)} not found")
        continue

    fname  = os.path.basename(csv_path)
    size_mb = os.path.getsize(csv_path) / 1024**2
    print(f"\n  Processing {fname} ({size_mb:.0f} MB) ...")

    try:
        df = pd.read_csv(
            csv_path,
            encoding="utf-16",         # FIX: quangbut CSVs are UTF-16 LE
            nrows=MAX_ROWS_PER_FILE,
            on_bad_lines="skip",
            low_memory=False
        )
        print(f"    Shape: {df.shape}, Columns: {list(df.columns)[:5]}")
    except Exception as e:
        print(f"    [WARN] utf-16 failed: {e}. Trying utf-8-sig...")
        try:
            df = pd.read_csv(
                csv_path, encoding="utf-8-sig",
                nrows=MAX_ROWS_PER_FILE, on_bad_lines="skip", low_memory=False
            )
        except Exception as e2:
            print(f"    [ERROR] Cannot read {fname}: {e2}")
            continue

    text_cols = get_text_columns(df)
    print(f"    Text columns detected: {text_cols}")

    if not text_cols:
        print(f"    [WARN] No text columns found, skipping.")
        continue

    for _, row in df[text_cols].iterrows():
        combined = " ".join(str(v) for v in row.values if pd.notna(v))
        cleaned  = clean_text(combined)
        if passes_quality_filter(cleaned, min_chars=80):
            cpt_quangbut.append({"text": cleaned, "source": fname})
        else:
            skipped += 1

    print(f"    Accepted rows: {len([r for r in cpt_quangbut if r['source']==fname]):,}")

n_quangbut = write_jsonl(cpt_quangbut, f"{CPT_DIR}/cpt_quangbut.jsonl")
print(f"\n  Total CPT records from quangbut: {n_quangbut:,}")
print(f"  Skipped: {skipped:,}")
print(f"  Saved -> {CPT_DIR}/cpt_quangbut.jsonl ({os.path.getsize(CPT_DIR+'/cpt_quangbut.jsonl')/1024**2:.2f} MB)")


  Processing vbpl.csv (1487 MB) ...
    Shape: (50000, 4), Columns: ['so_hieu', 'dieu', 'prefix', 'noi_dung_vb']
    Text columns detected: ['prefix', 'noi_dung_vb']
    Accepted rows: 49,418

  Processing legal_truncated_corpus.csv (1693 MB) ...
    Shape: (50000, 4), Columns: ['Unnamed: 0', 'id', 'title', 'context']
    Text columns detected: ['context']
    Accepted rows: 49,961

  Processing sent_truncated_vbpl_update.csv (1827 MB) ...
    Shape: (50000, 3), Columns: ['so_hieu', 'dieu', 'truncated_text']
    Text columns detected: ['truncated_text']
    Accepted rows: 49,893

  Processing sent_truncated_vbpl_legal_only.csv (16 MB) ...
    Shape: (9443, 4), Columns: ['so_hieu', 'dieu', 'prefix', 'truncated_text']
    Text columns detected: ['truncated_text']
    Accepted rows: 9,374

  Processing vbpl_crawl_2.csv (1534 MB) ...
    Shape: (50000, 8), Columns: ['id', 'ministry', 'type', 'name', 'chapter_id']
    Text columns detected: ['article', 'content']
    Accepted rows: 48,703


In [11]:
# ============================================================
# Cell 9 — Merge and split SFT datasets
# Merges all SFT sources, shuffles, splits 90/10 train/val.
# Output: sft/sft_train.jsonl, sft/sft_val.jsonl
# ============================================================
import json, os, random

random.seed(42)

all_sft = []
sft_sources = [
    f"{SFT_DIR}/sft_vn_legal_qa.jsonl",
    f"{SFT_DIR}/sft_vn_legal_chat.jsonl",
    f"{SFT_DIR}/sft_vibidlqa.jsonl",
]

print("Loading SFT sources:")
for path in sft_sources:
    if not os.path.exists(path):
        print(f"  [SKIP] {path}")
        continue
    before = len(all_sft)
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try: all_sft.append(json.loads(line))
            except: continue
    print(f"  {os.path.basename(path)}: +{len(all_sft)-before:,} records")

# Shuffle
random.shuffle(all_sft)

# 90/10 split
split_idx  = int(len(all_sft) * 0.9)
sft_train  = all_sft[:split_idx]
sft_val    = all_sft[split_idx:]

n_train = write_jsonl(sft_train, f"{SFT_DIR}/sft_train.jsonl")
n_val   = write_jsonl(sft_val,   f"{SFT_DIR}/sft_val.jsonl")

print(f"\n  Total SFT samples : {len(all_sft):,}")
print(f"  Train (90%)       : {n_train:,}  -> sft_train.jsonl")
print(f"  Val   (10%)       : {n_val:,}    -> sft_val.jsonl")

# Source distribution in train
from collections import Counter
src_dist = Counter(r.get("source", "unknown") for r in sft_train)
print("\n  Source distribution in train:")
for src, count in src_dist.most_common():
    print(f"    {src:<30}: {count:,}")

Loading SFT sources:
  sft_vn_legal_qa.jsonl: +29,141 records
  sft_vn_legal_chat.jsonl: +2,651 records
  sft_vibidlqa.jsonl: +1,928 records

  Total SFT samples : 33,720
  Train (90%)       : 30,348  -> sft_train.jsonl
  Val   (10%)       : 3,372    -> sft_val.jsonl

  Source distribution in train:
    vn_legal_qa                   : 26,223
    vn_legal_chat                 : 2,387
    ViBidLQA_v1                   : 1,738


In [12]:
# ============================================================
# Cell 10 — Merge CPT corpus
# Combines all CPT sources into cpt_corpus.jsonl.
# Format: {text: str, source: str}
# ============================================================
import json, os

CPT_SOURCES = [
    f"{CPT_DIR}/cpt_zalo.jsonl",
    f"{CPT_DIR}/cpt_vn_legal_qa.jsonl",
    f"{CPT_DIR}/cpt_large_queries.jsonl",
    f"{CPT_DIR}/cpt_quangbut.jsonl",
]

total_words = 0
total_count = 0

print("Merging CPT sources:")
with open(f"{CPT_DIR}/cpt_corpus.jsonl", "w", encoding="utf-8") as out_f:
    for path in CPT_SOURCES:
        if not os.path.exists(path):
            print(f"  [SKIP] {path}")
            continue
        src_count = 0
        src_words = 0
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    src_count += 1
                    src_words += len(rec.get("text", "").split())
                except:
                    continue
        total_count += src_count
        total_words += src_words
        print(f"  {os.path.basename(path):<35}: {src_count:>8,} records | {src_words:>12,} words")

cpt_size_mb = os.path.getsize(f"{CPT_DIR}/cpt_corpus.jsonl") / 1024**2
print(f"\n  TOTAL records : {total_count:,}")
print(f"  TOTAL words   : {total_words:,}")
print(f"  Est. BPE tokens: {int(total_words * 1.3):,}")
print(f"  File size     : {cpt_size_mb:.2f} MB")
print(f"  Saved -> {CPT_DIR}/cpt_corpus.jsonl")

Merging CPT sources:
  cpt_zalo.jsonl                     :   60,522 records |   16,749,502 words
  cpt_vn_legal_qa.jsonl              :    9,377 records |    2,336,232 words
  cpt_large_queries.jsonl            :  138,268 records |   34,502,163 words
  cpt_quangbut.jsonl                 :  207,349 records |   47,543,639 words

  TOTAL records : 415,516
  TOTAL words   : 101,131,536
  Est. BPE tokens: 131,470,996
  File size     : 606.22 MB
  Saved -> /kaggle/working/data_processed/cpt/cpt_corpus.jsonl


In [14]:
# ============================================================
# Cell 11 — Final summary and Stage 2 report
# ============================================================
import json, os
from datetime import datetime, timezone

def count_jsonl(path):
    if not os.path.exists(path):
        return 0
    return sum(1 for _ in open(path, encoding="utf-8"))

def size_mb(path):
    return round(os.path.getsize(path) / 1024**2, 2) if os.path.exists(path) else 0

outputs = {
    "cpt_corpus":                  f"{CPT_DIR}/cpt_corpus.jsonl",
    "sft_train":                   f"{SFT_DIR}/sft_train.jsonl",
    "sft_val":                     f"{SFT_DIR}/sft_val.jsonl",
    "dpo_train":                   f"{DPO_DIR}/dpo_train.jsonl",
    "rag_zalo_passages":           f"{RAG_DIR}/rag_zalo_passages.jsonl",
    "rag_large_queries_passages":  f"{RAG_DIR}/rag_large_queries_passages.jsonl",
    "rag_large_queries_pairs":     f"{RAG_DIR}/rag_large_queries_pairs.jsonl",
    "eval_gold":                   f"{EVAL_DIR}/eval_gold.jsonl",
}

print("=" * 60)
print("STAGE 2 OUTPUT SUMMARY")
print("=" * 60)
print(f"{'File':<35} {'Records':>8} {'Size MB':>10}")
print("-" * 58)
for label, path in outputs.items():
    n  = count_jsonl(path)
    mb = size_mb(path)
    ok = "OK" if n > 0 else "EMPTY"
    print(f"  {label:<33} {n:>8,} {mb:>9.2f} MB  [{ok}]")

sft_total  = count_jsonl(outputs["sft_train"]) + count_jsonl(outputs["sft_val"])
dpo_total  = count_jsonl(outputs["dpo_train"])
eval_total = count_jsonl(outputs["eval_gold"])

print()
print("ACCEPTANCE CRITERIA CHECK")
print(f"  SFT samples >= 10,000  : {sft_total:,}   {'PASS' if sft_total >= 10000 else 'FAIL'}")
print(f"  DPO pairs  >= 2,000    : {dpo_total:,}   {'PASS' if dpo_total >= 2000 else 'FAIL'}")
print(f"  Eval gold  >= 3,000    : {eval_total:,}  {'PASS' if eval_total >= 3000 else 'FAIL'}")

report = {
    "stage":     "02_data_preprocessing",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "outputs":   {k: {"path": v, "records": count_jsonl(v), "size_mb": size_mb(v)}
                  for k, v in outputs.items()},
    "criteria": {
        "sft_total": sft_total,
        "dpo_total": dpo_total,
        "eval_total": eval_total,
        "all_pass": sft_total >= 10000 and dpo_total >= 2000 and eval_total >= 3000
    }
}
rpath = f"{LOG_DIR}/stage02_preprocessing_report.json"
with open(rpath, "w", encoding="utf-8") as ff:
    json.dump(report, ff, ensure_ascii=False, indent=2)

print(f"\nReport saved: {rpath}")
print("\nNext: 03_cpt_training.ipynb")


STAGE 2 OUTPUT SUMMARY
File                                 Records    Size MB
----------------------------------------------------------
  cpt_corpus                         415,516    606.22 MB  [OK]
  sft_train                           30,348     87.46 MB  [OK]
  sft_val                              3,372      9.77 MB  [OK]
  dpo_train                            6,424     14.56 MB  [OK]
  rag_zalo_passages                   60,522    112.41 MB  [OK]
  rag_large_queries_passages         138,268    241.14 MB  [OK]
  rag_large_queries_pairs            499,901    115.08 MB  [OK]
  eval_gold                            3,013      3.54 MB  [OK]

ACCEPTANCE CRITERIA CHECK
  SFT samples >= 10,000  : 33,720   PASS
  DPO pairs  >= 2,000    : 6,424   PASS
  Eval gold  >= 3,000    : 3,013  PASS

Report saved: /kaggle/working/logs/stage02_preprocessing_report.json

Next: 03_cpt_training.ipynb
